# 15 - Business Value

Este notebook cubre PB-16 del Sprint 5: traducir metricas tecnicas del modelo a valor de negocio.

Se evalua el valor esperado por modelo, threshold y segmento usando una matriz costo-beneficio. Los costos son supuestos iniciales y deben validarse con el sponsor antes de tomar una decision de despliegue.

In [ ]:
from pathlib import Path
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "processed" / "test_processed.csv"
CLEAN_DATA_PATH = ROOT / "data" / "interim" / "dataset_clean.csv"
MODELS_DIR = ROOT / "models"

THRESHOLD_RESULTS_PATH = MODELS_DIR / "business_value_threshold_search.csv"
RECOMMENDATIONS_PATH = MODELS_DIR / "business_value_recommendations.csv"
GAIN_CURVE_PATH = MODELS_DIR / "business_value_gain_lift_curve.csv"

print("ROOT:", ROOT)
print("TEST:", DATA_PATH)

## 1. Carga de datos y modelos

Se usa el test real desbalanceado, no el dataset balanceado con SMOTE, porque el valor de negocio debe reflejar la proporcion real de compradores.

In [ ]:
TARGET_COL = "Revenue"

df_test = pd.read_csv(DATA_PATH)
X_test = df_test.drop(columns=[TARGET_COL])
y_test = df_test[TARGET_COL].astype(int)

annual_scale_factor = 1 / 0.20
if CLEAN_DATA_PATH.exists():
    annual_scale_factor = len(pd.read_csv(CLEAN_DATA_PATH)) / len(df_test)

MODEL_PATHS = {
    ("Random Forest", "final_test"): MODELS_DIR / "best_tuned_model.pkl",
    ("XGBoost", "base"): MODELS_DIR / "xgboost_base.pkl",
    ("XGBoost", "tuned"): MODELS_DIR / "tuned_xgboost.pkl",
    ("LightGBM", "base"): MODELS_DIR / "lightgbm_base.pkl",
    ("LightGBM", "tuned"): MODELS_DIR / "tuned_lightgbm.pkl",
}

models = {}
for key, path in MODEL_PATHS.items():
    if path.exists():
        models[key] = joblib.load(path)

for key, model in models.items():
    if hasattr(model, "feature_names_in_") and list(model.feature_names_in_) != list(X_test.columns):
        raise ValueError(f"Columnas incompatibles para {key}")

print("Test shape:", X_test.shape)
print("Compradores reales:", int(y_test.sum()), f"({y_test.mean():.2%})")
print("Factor de escalamiento anual estimado:", round(annual_scale_factor, 2))
print("Modelos cargados:", list(models.keys()))

## 2. Supuestos de costo-beneficio

Como el dataset solo tiene `Revenue` binario y no monto de compra, se usan supuestos iniciales. Deben ser revisados por el sponsor:

- TP: comprador correctamente identificado y accionado.
- FP: usuario contactado que no compra.
- FN: comprador no detectado, oportunidad perdida.
- TN: no comprador correctamente ignorado.

In [ ]:
cost_profiles = pd.DataFrame([
    {
        "scenario": "global_conservative",
        "visitor_type": "ALL",
        "benefit_tp": 100,
        "cost_fp": -20,
        "cost_fn": -80,
        "benefit_tn": 0,
        "business_goal": "Ranking global cuando contactar usuarios tiene costo relevante.",
    },
    {
        "scenario": "max_capture",
        "visitor_type": "ALL",
        "benefit_tp": 100,
        "cost_fp": -10,
        "cost_fn": -120,
        "benefit_tn": 0,
        "business_goal": "Capturar la mayor cantidad posible de compradores.",
    },
    {
        "scenario": "new_visitors",
        "visitor_type": "New_Visitor",
        "benefit_tp": 120,
        "cost_fp": -35,
        "cost_fn": -90,
        "benefit_tn": 0,
        "business_goal": "Contactar nuevos usuarios solo cuando hay alta probabilidad de compra.",
    },
    {
        "scenario": "returning_visitors",
        "visitor_type": "Returning_Visitor",
        "benefit_tp": 80,
        "cost_fp": -5,
        "cost_fn": -70,
        "benefit_tn": 0,
        "business_goal": "Activar recurrentes con menor costo de contacto y mayor tolerancia a FP.",
    },
])

display(cost_profiles)

## 3. Busqueda de umbral optimo

Se calcula el valor esperado para thresholds de 0.05 a 0.95. El mejor threshold no necesariamente maximiza F1; maximiza valor de negocio bajo los supuestos definidos.

In [ ]:
def get_segment_mask(X, visitor_type):
    if visitor_type == "ALL":
        return pd.Series(True, index=X.index)
    col = f"cat__VisitorType_{visitor_type}"
    if col not in X.columns:
        raise ValueError(f"No existe la columna {col}")
    return X[col] == 1


def expected_value(y_true, y_pred, profile):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    value = (
        tp * profile["benefit_tp"]
        + fp * profile["cost_fp"]
        + fn * profile["cost_fn"]
        + tn * profile["benefit_tn"]
    )
    return int(tn), int(fp), int(fn), int(tp), float(value)


thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)
rows = []

for _, profile in cost_profiles.iterrows():
    mask = get_segment_mask(X_test, profile["visitor_type"])
    X_scope = X_test.loc[mask]
    y_scope = y_test.loc[mask]

    if len(y_scope) == 0:
        continue

    baseline_pred = np.zeros(len(y_scope), dtype=int)
    _, _, baseline_fn, _, baseline_value = expected_value(y_scope, baseline_pred, profile)

    for (model_name, version), model in models.items():
        y_proba = model.predict_proba(X_scope)[:, 1]

        for threshold in thresholds:
            y_pred = (y_proba >= threshold).astype(int)
            tn, fp, fn, tp, value = expected_value(y_scope, y_pred, profile)

            rows.append({
                "scenario": profile["scenario"],
                "visitor_type": profile["visitor_type"],
                "model": model_name,
                "version": version,
                "threshold": threshold,
                "segment_size": len(y_scope),
                "buyers": int(y_scope.sum()),
                "predicted_positive": int(y_pred.sum()),
                "tn": tn,
                "fp": fp,
                "fn": fn,
                "tp": tp,
                "precision": precision_score(y_scope, y_pred, zero_division=0),
                "recall": recall_score(y_scope, y_pred, zero_division=0),
                "f1": f1_score(y_scope, y_pred, zero_division=0),
                "roc_auc": roc_auc_score(y_scope, y_proba) if y_scope.nunique() == 2 else np.nan,
                "pr_auc": average_precision_score(y_scope, y_proba) if y_scope.nunique() == 2 else np.nan,
                "expected_value_test": value,
                "baseline_no_action_value_test": baseline_value,
                "incremental_value_test": value - baseline_value,
                "estimated_annual_value": value * annual_scale_factor,
                "estimated_annual_incremental_value": (value - baseline_value) * annual_scale_factor,
            })

threshold_results = pd.DataFrame(rows)
threshold_results.to_csv(THRESHOLD_RESULTS_PATH, index=False)

print("Resultados de threshold guardados en:", THRESHOLD_RESULTS_PATH)
display(threshold_results.head())

In [ ]:
recommendations = (
    threshold_results
    .sort_values(["scenario", "expected_value_test", "pr_auc", "precision"], ascending=[True, False, False, False])
    .groupby("scenario", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

recommendations = recommendations.merge(
    cost_profiles[["scenario", "business_goal", "benefit_tp", "cost_fp", "cost_fn", "benefit_tn"]],
    on="scenario",
    how="left",
)

recommendations.to_csv(RECOMMENDATIONS_PATH, index=False)

cols = [
    "scenario", "visitor_type", "model", "version", "threshold", "business_goal",
    "expected_value_test", "incremental_value_test", "estimated_annual_incremental_value",
    "precision", "recall", "f1", "tp", "fp", "fn", "predicted_positive",
]

display(recommendations[cols])
print("Recomendaciones guardadas en:", RECOMMENDATIONS_PATH)

## 4. Curva de ganancia y lift

La curva de ganancia muestra cuantos compradores se capturan al contactar el top X% de sesiones segun probabilidad predicha. El lift compara esa captura contra una seleccion aleatoria.

In [ ]:
ranking_model_key = ("XGBoost", "tuned")
if ranking_model_key not in models:
    ranking_model_key = next(iter(models.keys()))

ranking_model = models[ranking_model_key]
ranking_proba = ranking_model.predict_proba(X_test)[:, 1]

gain_df = pd.DataFrame({
    "y_true": y_test.values,
    "proba": ranking_proba,
}).sort_values("proba", ascending=False).reset_index(drop=True)

gain_df["rank"] = np.arange(1, len(gain_df) + 1)
gain_df["contact_rate"] = gain_df["rank"] / len(gain_df)
gain_df["cum_buyers"] = gain_df["y_true"].cumsum()
gain_df["gain"] = gain_df["cum_buyers"] / gain_df["y_true"].sum()
gain_df["random_gain"] = gain_df["contact_rate"]
gain_df["lift"] = gain_df["gain"] / gain_df["random_gain"]
gain_df["model"] = ranking_model_key[0]
gain_df["version"] = ranking_model_key[1]

gain_df.to_csv(GAIN_CURVE_PATH, index=False)
print("Curva gain/lift guardada en:", GAIN_CURVE_PATH)

display(gain_df.loc[gain_df["contact_rate"].isin(gain_df["contact_rate"]), ["contact_rate", "gain", "lift"]].iloc[[int(len(gain_df)*p)-1 for p in [0.1, 0.2, 0.3, 0.5]]])

plt.figure(figsize=(8, 5))
plt.plot(gain_df["contact_rate"], gain_df["gain"], label=f"{ranking_model_key[0]} {ranking_model_key[1]}")
plt.plot(gain_df["contact_rate"], gain_df["random_gain"], linestyle="--", label="Baseline aleatoria")
plt.xlabel("Porcentaje de sesiones contactadas")
plt.ylabel("Porcentaje de compradores capturados")
plt.title("Curva de ganancia acumulada")
plt.legend()
plt.grid(True)
plt.show()

## 5. Recomendacion BA

La decision recomendada es no usar un unico umbral global para todos los usuarios. Se propone una politica por escenario:

- Campanas globales con costo de contacto: usar el modelo/umbral que maximiza valor en `global_conservative`.
- Objetivo de maxima captura: usar la politica de `max_capture`.
- Usuarios nuevos: usar una politica mas conservadora por mayor costo de contacto.
- Usuarios recurrentes: usar una politica mas agresiva porque la activacion suele ser mas barata.

Los valores economicos deben presentarse como estimaciones bajo supuestos. Antes de Sprint 6, el sponsor debe validar `benefit_tp`, `cost_fp` y `cost_fn`.